# Responsiveness checks — M2-5, M2-6

Both cases are "does this server keep answering OTHER requests while one request is doing real CPU-bound
work" — a fix verified by TIMING/concurrency behavior, not by a stored value. `asyncio.to_thread()` moves
the blocking work off the single event loop; these notebooks prove that by racing a "heartbeat" coroutine
against the real service call and checking the heartbeat wasn't starved.

**Real-world scenario common to both:** this server runs as one `uvicorn` worker with one event loop
serving every in-flight request. Decrypting a data push (M2-5) or assembling FHIR bundles for a large
health information request (M2-6) is real CPU-bound work. If that work runs directly inside the `async def`
handler (not off-loaded), it blocks the ENTIRE event loop for however long it takes — every other request
this server is handling at that moment (a `/health` check, an unrelated callback, anything) stalls until it
finishes. `asyncio.to_thread()` moves the blocking work to a worker thread so the event loop stays free to
keep serving everything else.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import harness


---
## M2-5 — decrypting a data push must not freeze the event loop

**Pass criteria:** while a ~0.6s (simulated) decrypt is running inside `process_health_information_hiu_push()`,
a concurrently-scheduled heartbeat coroutine keeps ticking (proving the event loop was never blocked).

In [ ]:
import asyncio
import time
from unittest.mock import patch

import server.callbacks.services.health_information_hiu_push_service as push_service
from server.callbacks.repository.pending_health_information_request_repository import (
    save_pending_health_information_request, link_transaction_id,
)
from server.callbacks.repository.hiu_consent_repository import save_hiu_consent

harness.activate_scratch_storage("m2_5")

save_pending_health_information_request("req-resp-1", {
    "consent_id": "consent-resp", "hip_id": "IN2810000123", "hiu_id": "HIU-1",
    "key_material": {"private_key": "p", "nonce": "n"},
})
link_transaction_id("req-resp-1", "txn-resp-1")
save_hiu_consent("consent-resp", {"consent_detail": {"careContexts": [{"careContextReference": "cc-1"}]}})

def slow_decrypt(ciphertext, **kw):
    time.sleep(0.6)  # stands in for real CPU-bound RSA/ECDH + AES-GCM decryption work
    return '{"resourceType": "Bundle"}'

heartbeat = {"count": 0}
async def heartbeat_loop():
    for _ in range(20):
        await asyncio.sleep(0.05)
        heartbeat["count"] += 1

with patch.object(push_service, "decrypt_health_data", slow_decrypt), \
     patch.object(push_service, "from_x509_public_key", lambda v: "FAKE"), \
     patch.object(push_service, "send_health_information_notify", lambda **kw: harness.FakeResponse(202)):
    push_body = {
        "transactionId": "txn-resp-1", "pageNumber": 0, "pageCount": 1,
        "entries": [{"content": "c", "checksum": push_service._compute_checksum("c"), "careContextReference": "cc-1"}],
        "keyMaterial": {"dhPublicKey": {"keyValue": "k"}, "nonce": "n"},
    }
    hb_task = asyncio.create_task(heartbeat_loop())
    await push_service.process_health_information_hiu_push({"body": push_body})
    await asyncio.sleep(0.1)
    hb_task.cancel()

harness.check(f"event loop kept ticking during the ~0.6s decrypt (heartbeat incremented {heartbeat['count']} times, expected >5)", heartbeat["count"] > 5)


---
## M2-6 — assembling FHIR bundles must not freeze the event loop

**Pass criteria:** same heartbeat-racing approach, this time around `build_bundles_for_care_contexts()`
inside `process_health_information_request()`.

In [ ]:
import server.callbacks.services.health_information_request_service as request_service
from server.callbacks.repository.consent_repository import save_consent

harness.activate_scratch_storage("m2_6")

save_consent("consent-resp-2", {"care_contexts": [{"careContextReference": "cc-1"}]})

def slow_build_bundles(care_context_references, date_range=None):
    time.sleep(0.6)
    return {ref: {"resourceType": "Bundle"} for ref in care_context_references}

push_notify_calls = []
def fake_push_and_notify(**kw):
    push_notify_calls.append(kw)

heartbeat2 = {"count": 0}
async def heartbeat_loop2():
    for _ in range(20):
        await asyncio.sleep(0.05)
        heartbeat2["count"] += 1

with patch.object(request_service, "build_bundles_for_care_contexts", slow_build_bundles), \
     patch.object(request_service, "_push_and_notify", fake_push_and_notify), \
     patch.object(request_service, "send_on_health_information_request", lambda **kw: harness.FakeResponse(200)):
    callback_data = {
        "headers": {"request-id": "req-resp-2", "x-hip-id": "IN2810000123"},
        "body": {"transactionId": "txn-resp-2", "hiRequest": {
            "consent": {"id": "consent-resp-2"}, "dataPushUrl": "https://example/push",
            "keyMaterial": {"dhPublicKey": {"keyValue": "k"}, "nonce": "n"},
        }},
    }
    hb_task2 = asyncio.create_task(heartbeat_loop2())
    await request_service.process_health_information_request(callback_data)
    await asyncio.sleep(0.1)
    hb_task2.cancel()

harness.check(f"event loop kept ticking during the ~0.6s bundle assembly (heartbeat incremented {heartbeat2['count']} times, expected >5)", heartbeat2["count"] > 5)
harness.check("the assembled bundles still correctly flowed through to the push step afterward", len(push_notify_calls) == 1)
